# What does each country’s gender-risk profile look like when we integrate structure, dynamics, shocks, and observability?

In [2]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.getcwd()).resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(os.getcwd())

/Users/daniehbenotman/Desktop/worldbank_tableau_project/analysis/synthesis


In [4]:
os.chdir("/Users/daniehbenotman/Desktop/worldbank_tableau_project")
print("✅ Working directory changed to:", os.getcwd())

✅ Working directory changed to: /Users/daniehbenotman/Desktop/worldbank_tableau_project


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


# Display
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Reproducibility
RANDOM_STATE = 42


In [15]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/daniehbenotman/Desktop/worldbank_tableau_project")

CLASSIFICATION_DIR = PROJECT_ROOT / "analysis" / "analysis_outputs" / "classification"
CLUSTERING_DIR     = PROJECT_ROOT / "analysis" / "analysis_outputs" / "clustering"
ANOMALY_DIR        = PROJECT_ROOT / "analysis" / "analysis_outputs" / "anomaly"
CAVEATS_DIR        = PROJECT_ROOT / "analysis" / "methodological_caveats"

OUTPUT_DIR = PROJECT_ROOT / "analysis" / "analysis_outputs" / "synthesis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



In [16]:
classification_tiers = pd.read_csv(
    CLASSIFICATION_DIR / "classification_country_tiers.csv"
)

classification_shap_drivers = pd.read_csv(
    CLASSIFICATION_DIR / "classification_SHAP_top_drivers.csv"
)

classification_shap_direction = pd.read_csv(
    CLASSIFICATION_DIR / "classification_SHAP_directionality.csv"
)


In [18]:
structural_clusters = pd.read_csv(
    CLUSTERING_DIR / "structural_clusters.csv"
)

structural_clusters = structural_clusters.rename(columns={
    "country_name": "country",
    "cluster": "structural_cluster"
})


In [26]:
trajectory_summary = pd.read_csv(
    CLUSTERING_DIR / "trajectory_clusters.csv"
)

trajectory_summary = trajectory_summary.rename(columns={
    "country_name": "country"
})


In [29]:
anomaly_scores = pd.read_csv(
    ANOMALY_DIR / "country_anomaly_scores.csv"
)

anomaly_scores = anomaly_scores.rename(columns={
    "country_name": "country"
})
#anomaly_scores.head()

#### Creating Data Availabity Labels

In [83]:
data_observability_map = {
    # Very High
    "United Arab Emirates": "Very High",
    "Qatar": "High",
    "Bahrain": "High",
    "Oman": "High",
    "Saudi Arabia": "High",
    "Jordan": "High",
    "Morocco": "High",

    # Medium
    "Algeria": "Medium",
    "Kuwait": "Medium",
    "Tunisia": "Medium-High",
    "Egypt, Arab Rep.": "Medium-Low",
    "Iran, Islamic Rep.": "Medium-Low",
    "Lebanon": "Medium-Low",

    # Low
    "Iraq": "Low",
    "West Bank and Gaza": "Very Low",

    # Collapsed
    "Syrian Arab Republic": "Collapsed",
    "Yemen, Rep.": "Collapsed",
    "Libya": "Collapsed"
}


In [84]:
data_observability = (
    pd.DataFrame.from_dict(
        data_observability_map,
        orient="index",
        columns=["data_reliability"]
    )
    .reset_index()
    .rename(columns={"index": "country"})
)
data_observability

,country,data_reliability
0,United Arab Emirates,Very High
1,Qatar,High
2,Bahrain,High
3,Oman,High
4,Saudi Arabia,High
5,Jordan,High
6,Morocco,High
7,Algeria,Medium
8,Kuwait,Medium
9,Tunisia,Medium-High


In [85]:
data_observability.to_csv(
    OUTPUT_DIR / "data_observability_labels.csv",
    index=False
)


#### Choosing the Columns from Trajectory

In [28]:
trajectory_summary.columns

Index(['country', 'cat_Demographics_trend_slope', 'cat_Economy_trend_slope', 'cat_Education_trend_slope',
       'cat_Governance_trend_slope', 'cat_Health_trend_slope', 'cat_Labor_trend_slope',
       'cat_LegalAutonomy_trend_slope', 'cat_WBL_trend_slope', 'cat_Demographics_cagr', 'cat_Economy_cagr',
       'cat_Education_cagr', 'cat_Governance_cagr', 'cat_Health_cagr', 'cat_Labor_cagr', 'cat_WBL_cagr',
       'cat_Demographics_momentum', 'cat_Economy_momentum', 'cat_Education_momentum', 'cat_Governance_momentum',
       'cat_Health_momentum', 'cat_Labor_momentum', 'cat_LegalAutonomy_momentum', 'cat_WBL_momentum',
       'cat_Demographics_rolling_std_5y', 'cat_Economy_rolling_std_5y', 'cat_Education_rolling_std_5y',
       'cat_Governance_rolling_std_5y', 'cat_Health_rolling_std_5y', 'cat_Labor_rolling_std_5y',
       'cat_WBL_rolling_std_5y', 'cat_Demographics_spike', 'cat_Economy_spike', 'cat_Education_spike',
       'cat_Governance_spike', 'cat_Health_spike', 'cat_Labor_spike', 'cat

In [31]:
coef_var_cols = [
    "cat_Demographics_coef_variation",
    "cat_Economy_coef_variation",
    "cat_Education_coef_variation",
    "cat_Governance_coef_variation",
    "cat_Health_coef_variation",
    "cat_Labor_coef_variation",
    "cat_LegalAutonomy_coef_variation",
    "cat_WBL_coef_variation"
]

trajectory_summary["dominant_trajectory_domain"] = (
    trajectory_summary[coef_var_cols]
    .idxmax(axis=1)
    .str.replace("cat_", "", regex=False)
    .str.replace("_coef_variation", "", regex=False)
)


In [32]:
trajectory_summary_for_synthesis = (
    trajectory_summary[[
        "country",
        "Cluster_Name",
        "dominant_trajectory_domain"
    ]]
    .rename(columns={
        "Cluster_Name": "trajectory_cluster"
    })
)
trajectory_summary_for_synthesis.head(10)

,country,trajectory_cluster,dominant_trajectory_domain
0,Algeria,Education-Led Social Improvers,Labor
1,Bahrain,Steady Reformers,Economy
2,"Egypt, Arab Rep.",Education-Led Social Improvers,WBL
3,"Iran, Islamic Rep.",Education-Led Social Improvers,Education
4,Iraq,Education-Led Social Improvers,Education
5,Jordan,Steady Moderate Improvers,Governance
6,Kuwait,Steady Moderate Improvers,Health
7,Lebanon,Education-Led Social Improvers,Economy
8,Libya,Education-Led Social Improvers,Education
9,Morocco,Morocco,Health


In [34]:
trajectory_summary_for_synthesis[
    trajectory_summary_for_synthesis["country"]
    .isin(["Morocco", "Libya", "Lebanon", "Yemen"])
]



,country,trajectory_cluster,dominant_trajectory_domain
7,Lebanon,Education-Led Social Improvers,Economy
8,Libya,Education-Led Social Improvers,Education
9,Morocco,Morocco,Health


In [35]:
trajectory_summary_for_synthesis.to_csv(
    OUTPUT_DIR / "trajectory_summary_for_synthesis.csv",
    index=False
)


## Merging all Imports 

In [37]:
classification_tiers.head()


,country,risk_tier
0,Algeria,T0_reference
1,Bahrain,T0_reference
2,"Egypt, Arab Rep.",T0_reference
3,"Iran, Islamic Rep.",T0_reference
4,Iraq,T0_reference


In [38]:
set(classification_tiers["country"]) - set(trajectory_summary_for_synthesis["country"])


set()

In [39]:
set(trajectory_summary_for_synthesis["country"]) - set(classification_tiers["country"])


set()

In [41]:
cross_matrix_step1 = (
    classification_tiers
    .merge(
        trajectory_summary_for_synthesis,
        on="country",
        how="left"
    )
)
cross_matrix_step1.head(10)

,country,risk_tier,trajectory_cluster,dominant_trajectory_domain
0,Algeria,T0_reference,Education-Led Social Improvers,Labor
1,Bahrain,T0_reference,Steady Reformers,Economy
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education
4,Iraq,T0_reference,Education-Led Social Improvers,Education
5,Jordan,T0_reference,Steady Moderate Improvers,Governance
6,Kuwait,T0_reference,Steady Moderate Improvers,Health
7,Lebanon,T3_strong,Education-Led Social Improvers,Economy
8,Libya,T0_reference,Education-Led Social Improvers,Education
9,Morocco,T3_strong,Morocco,Health


In [44]:
cross_matrix_step1.isna().sum()



country                       0
risk_tier                     0
trajectory_cluster            0
dominant_trajectory_domain    0
dtype: int64

In [45]:
structural_clusters.head()

,country,Cluster,Cluster_Name
0,Algeria,3,North African Reformers
1,Bahrain,1,High-Income Empowerers
2,"Egypt, Arab Rep.",2,Fragile States
3,"Iran, Islamic Rep.",2,Fragile States
4,Iraq,2,Fragile States


In [47]:
structural_clusters = structural_clusters.rename(columns={
    "Cluster_Name": "structural_cluster"
})


In [49]:
set(structural_clusters["country"]) - set(cross_matrix_step1["country"])

set()

In [51]:
cross_matrix_step2 = (
    cross_matrix_step1
    .merge(
        structural_clusters[["country", "structural_cluster"]],
        on="country",
        how="left"
    )
)
cross_matrix_step2.head(10)

,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster
0,Algeria,T0_reference,Education-Led Social Improvers,Labor,North African Reformers
1,Bahrain,T0_reference,Steady Reformers,Economy,High-Income Empowerers
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL,Fragile States
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education,Fragile States
4,Iraq,T0_reference,Education-Led Social Improvers,Education,Fragile States
5,Jordan,T0_reference,Steady Moderate Improvers,Governance,Fragile States
6,Kuwait,T0_reference,Steady Moderate Improvers,Health,High-Income Empowerers
7,Lebanon,T3_strong,Education-Led Social Improvers,Economy,Lebanon Outlier
8,Libya,T0_reference,Education-Led Social Improvers,Education,North African Reformers
9,Morocco,T3_strong,Morocco,Health,North African Reformers


In [53]:
cross_matrix_step2.isna().sum()


country                       0
risk_tier                     0
trajectory_cluster            0
dominant_trajectory_domain    0
structural_cluster            0
dtype: int64

In [56]:
anomaly_scores.head()

,country,Cluster,Cluster_Name,PC1,PC2,PC3,iso_raw_score,iso_anomaly_score,iso_label
0,Lebanon,0,Lebanon Outlier,-0.729063,3.136066,2.908014,-0.089973,0.089973,1
1,West Bank and Gaza,4,Palestine,-0.624159,-3.355717,1.521529,-0.045399,0.045399,1
2,Morocco,3,North African Reformers,-1.118704,1.106070,-2.427395,-0.015850,0.015850,1
3,Qatar,1,High-Income Empowerers,3.944395,-0.583992,0.376699,0.012968,-0.012968,0
4,"Yemen, Rep.",2,Fragile States,-2.902448,-1.372818,-0.466374,0.015127,-0.015127,0


In [58]:
anomaly_summary_for_synthesis = (
    anomaly_scores[[
        "country",
        "iso_anomaly_score"
    ]]
    .rename(columns={
        "iso_anomaly_score": "shock_sensitivity"
    })
)
anomaly_summary_for_synthesis.head()



,country,shock_sensitivity
0,Lebanon,0.089973
1,West Bank and Gaza,0.045399
2,Morocco,0.015850
3,Qatar,-0.012968
4,"Yemen, Rep.",-0.015127


In [59]:
cross_matrix_step3 = (
    cross_matrix_step2
    .merge(
        anomaly_summary_for_synthesis,
        on="country",
        how="left"
    )
)
cross_matrix_step3.isna().sum()

country                       0
risk_tier                     0
trajectory_cluster            0
dominant_trajectory_domain    0
structural_cluster            0
shock_sensitivity             0
dtype: int64

In [62]:
classification_shap_drivers

,country,driver_1,driver_2,driver_3
0,Lebanon,cat_Demographics_cagr,cat_LegalAutonomy_coef_variation,regional_deviation_std_mean
1,Morocco,cat_LegalAutonomy_coef_variation,cat_Economy_trend_slope,coef_variation_mean
2,West Bank and Gaza,cat_Demographics_cagr,cat_LegalAutonomy_coef_variation,coef_variation_mean


In [63]:
cross_matrix_step4 = (
    cross_matrix_step3
    .merge(
        classification_shap_drivers,
        on="country",
        how="left"
    )
)
cross_matrix_step4.head(10)


,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster,shock_sensitivity,driver_1,driver_2,driver_3
0,Algeria,T0_reference,Education-Led Social Improvers,Labor,North African Reformers,-0.087545,NaN,NaN,NaN
1,Bahrain,T0_reference,Steady Reformers,Economy,High-Income Empowerers,-0.134814,NaN,NaN,NaN
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL,Fragile States,-0.149559,NaN,NaN,NaN
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.120607,NaN,NaN,NaN
4,Iraq,T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.125821,NaN,NaN,NaN
5,Jordan,T0_reference,Steady Moderate Improvers,Governance,Fragile States,-0.153246,NaN,NaN,NaN
6,Kuwait,T0_reference,Steady Moderate Improvers,Health,High-Income Empowerers,-0.112110,NaN,NaN,NaN
7,Lebanon,T3_strong,Education-Led Social Improvers,Economy,Lebanon Outlier,0.089973,cat_Demographics_cagr,cat_LegalAutonomy_coef_variation,regional_deviation_std_mean
8,Libya,T0_reference,Education-Led Social Improvers,Education,North African Reformers,-0.068374,NaN,NaN,NaN
9,Morocco,T3_strong,Morocco,Health,North African Reformers,0.015850,cat_LegalAutonomy_coef_variation,cat_Economy_trend_slope,coef_variation_mean


In [64]:
cross_matrix_step4[
    cross_matrix_step4["country"]
    .isin(["Morocco", "Lebanon", "Qatar", "Libya", "Yemen"])
]


,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster,shock_sensitivity,driver_1,driver_2,driver_3
7,Lebanon,T3_strong,Education-Led Social Improvers,Economy,Lebanon Outlier,0.089973,cat_Demographics_cagr,cat_LegalAutonomy_coef_variation,regional_deviation_std_mean
8,Libya,T0_reference,Education-Led Social Improvers,Education,North African Reformers,-0.068374,NaN,NaN,NaN
9,Morocco,T3_strong,Morocco,Health,North African Reformers,0.015850,cat_LegalAutonomy_coef_variation,cat_Economy_trend_slope,coef_variation_mean
11,Qatar,T0_reference,Steady Moderate Improvers,Demographics,High-Income Empowerers,-0.012968,NaN,NaN,NaN


In [65]:
cross_matrix_step4.isna().sum()


country                        0
risk_tier                      0
trajectory_cluster             0
dominant_trajectory_domain     0
structural_cluster             0
shock_sensitivity              0
driver_1                      15
driver_2                      15
driver_3                      15
dtype: int64

In [68]:
classification_shap_direction.head()


,country,feature,direction
0,Lebanon,cat_Demographics_cagr,protective
1,Lebanon,cat_LegalAutonomy_coef_variation,risk_increasing
2,Lebanon,regional_deviation_std_mean,risk_increasing
3,Morocco,cat_LegalAutonomy_coef_variation,risk_increasing
4,Morocco,cat_Economy_trend_slope,risk_increasing


In [70]:
classification_shap_direction["country"].unique()


array(['Lebanon', 'Morocco', 'West Bank and Gaza'], dtype=object)

In [71]:
shap_directionality_wide = (
    classification_shap_direction
    .assign(driver_rank=lambda df: df.groupby("country").cumcount() + 1)
    .pivot(
        index="country",
        columns="driver_rank",
        values="direction"
    )
    .rename(columns={
        1: "driver_1_direction",
        2: "driver_2_direction",
        3: "driver_3_direction"
    })
    .reset_index()
)
shap_directionality_wide

driver_rank,country,driver_1_direction,driver_2_direction,driver_3_direction
0,Lebanon,protective,risk_increasing,risk_increasing
1,Morocco,risk_increasing,risk_increasing,risk_increasing
2,West Bank and Gaza,protective,risk_increasing,risk_increasing


In [72]:
cross_matrix_step5 = (
    cross_matrix_step4
    .merge(
        shap_directionality_wide,
        on="country",
        how="left"
    )
)


In [73]:
cross_matrix_step5[
    cross_matrix_step5["risk_tier"] == "T3_strong"
][[
    "country",
    "driver_1", "driver_1_direction",
    "driver_2", "driver_2_direction",
    "driver_3", "driver_3_direction"
]]


,country,driver_1,driver_1_direction,driver_2,driver_2_direction,driver_3,driver_3_direction
7,Lebanon,cat_Demographics_cagr,protective,cat_LegalAutonomy_coef_variation,risk_increasing,regional_deviation_std_mean,risk_increasing
9,Morocco,cat_LegalAutonomy_coef_variation,risk_increasing,cat_Economy_trend_slope,risk_increasing,coef_variation_mean,risk_increasing
16,West Bank and Gaza,cat_Demographics_cagr,protective,cat_LegalAutonomy_coef_variation,risk_increasing,coef_variation_mean,risk_increasing


In [74]:
cross_matrix_step5[
    cross_matrix_step5["risk_tier"] != "T3_strong"
][[
    "country",
    "driver_1",
    "driver_1_direction"
]].head()


,country,driver_1,driver_1_direction
0,Algeria,NaN,NaN
1,Bahrain,NaN,NaN
2,"Egypt, Arab Rep.",NaN,NaN
3,"Iran, Islamic Rep.",NaN,NaN
4,Iraq,NaN,NaN


In [75]:
cross_matrix_step5.isna().sum()


country                        0
risk_tier                      0
trajectory_cluster             0
dominant_trajectory_domain     0
structural_cluster             0
shock_sensitivity              0
driver_1                      15
driver_2                      15
driver_3                      15
driver_1_direction            15
driver_2_direction            15
driver_3_direction            15
dtype: int64

In [76]:
final_cross_model_matrix = cross_matrix_step5


In [77]:
final_cross_model_matrix.head(5)

,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster,shock_sensitivity,driver_1,driver_2,driver_3,driver_1_direction,driver_2_direction,driver_3_direction
0,Algeria,T0_reference,Education-Led Social Improvers,Labor,North African Reformers,-0.087545,NaN,NaN,NaN,NaN,NaN,NaN
1,Bahrain,T0_reference,Steady Reformers,Economy,High-Income Empowerers,-0.134814,NaN,NaN,NaN,NaN,NaN,NaN
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL,Fragile States,-0.149559,NaN,NaN,NaN,NaN,NaN,NaN
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.120607,NaN,NaN,NaN,NaN,NaN,NaN
4,Iraq,T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.125821,NaN,NaN,NaN,NaN,NaN,NaN


In [86]:
final_cross_model_matrix = final_cross_model_matrix.merge(
    data_observability,
    on="country",
    how= 'left'
)

In [87]:
final_cross_model_matrix.head()

,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster,shock_sensitivity,driver_1,driver_2,driver_3,driver_1_direction,driver_2_direction,driver_3_direction,data_reliability_x,data_reliability_y
0,Algeria,T0_reference,Education-Led Social Improvers,Labor,North African Reformers,-0.087545,NaN,NaN,NaN,NaN,NaN,NaN,Medium,Medium
1,Bahrain,T0_reference,Steady Reformers,Economy,High-Income Empowerers,-0.134814,NaN,NaN,NaN,NaN,NaN,NaN,High,High
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL,Fragile States,-0.149559,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medium-Low
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.120607,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Medium-Low
4,Iraq,T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.125821,NaN,NaN,NaN,NaN,NaN,NaN,Low,Low


In [89]:
final_cross_model_matrix.drop(columns='data_reliability_x',inplace=True)

In [91]:
final_cross_model_matrix.rename(columns={"data_reliability_y":"data_reliability"},inplace= True)

In [92]:
final_cross_model_matrix.isna().sum()

country                        0
risk_tier                      0
trajectory_cluster             0
dominant_trajectory_domain     0
structural_cluster             0
shock_sensitivity              0
driver_1                      15
driver_2                      15
driver_3                      15
driver_1_direction            15
driver_2_direction            15
driver_3_direction            15
data_reliability               0
dtype: int64

In [94]:
final_cross_model_matrix.to_csv('data/cross_model.csv')

In [111]:
def assign_archetype(row):
    
    fragile_structures = ["Fragile States", "Lebanon Outlier",'Palestine']
    
    # Rule 1 — Collapsed systems
    if row["data_reliability"] == "Collapsed":
        return "Collapsed Statistical System (False Stability)"
    
    # Rule 2 — Shock-exposed fragile systems
    if (
        row["risk_tier"] == "T3_strong"
        and row["structural_cluster"] in fragile_structures
        and row["shock_sensitivity"] > 0
    ):
        return "Shock-Exposed Fragile System"
    
    # Rule 3 — Reform–shock collision states
    if (
        row["risk_tier"] == "T3_strong"
        and row["structural_cluster"] != "Fragile States"
        and row["data_reliability"] in ["High", "Very High"]
    ):
        return "Reform–Shock Collision State"
    
    # Rule 4 — Structurally fragile but statistically buffered
    if (
        row["risk_tier"] == "T0_reference"
        and row["structural_cluster"] == "Fragile States"
        and row["data_reliability"] in ["Medium-Low", "Low"]
    ):
        return "Structurally Fragile but Statistically Buffered"
    
    # Rule 5 — Education-led improvers
    if (
        row["risk_tier"] == "T0_reference"
        and row["dominant_trajectory_domain"] == "Education"
        and row["data_reliability"] in ["Medium", "High", "Very High"]
    ):
        return "Education-Led Improvers with Latent Risk"
    
    # Rule 6 — Stable reformers
    if (
        row["risk_tier"] == "T0_reference"
        and row["shock_sensitivity"] <= 0
        and row["data_reliability"] in ["High", "Very High"]
    ):
        return "Stable Reformers with Institutional Momentum"
    
    # Rule 7 — Fallback
    return "Transitional or Mixed Trajectory System"


In [112]:
final_cross_model_matrix['country_archetype'] = final_cross_model_matrix.apply(assign_archetype, axis=1)

In [113]:
final_cross_model_matrix["country_archetype"].value_counts()


country_archetype
Stable Reformers with Institutional Momentum       6
Transitional or Mixed Trajectory System            3
Structurally Fragile but Statistically Buffered    3
Collapsed Statistical System (False Stability)     3
Shock-Exposed Fragile System                       2
Reform–Shock Collision State                       1
Name: count, dtype: int64

In [109]:
final_cross_model_matrix[
    final_cross_model_matrix["country"].isin([
        "Morocco", "Lebanon", "Egypt", "Libya", "Yemen", "Qatar"
    ])
][[
    "country",
    "risk_tier",
    "structural_cluster",
    "dominant_trajectory_domain",
    "shock_sensitivity",
    "data_reliability",
    "country_archetype"
]]


,country,risk_tier,structural_cluster,dominant_trajectory_domain,shock_sensitivity,data_reliability,country_archetype
7,Lebanon,T3_strong,Lebanon Outlier,Economy,0.089973,Medium-Low,Shock-Exposed Fragile System
8,Libya,T0_reference,North African Reformers,Education,-0.068374,Collapsed,Collapsed Statistical System (False Stability)
9,Morocco,T3_strong,North African Reformers,Health,0.015850,High,Reform–Shock Collision State
11,Qatar,T0_reference,High-Income Empowerers,Demographics,-0.012968,High,Stable Reformers with Institutional Momentum


In [114]:
final_cross_model_matrix

,country,risk_tier,trajectory_cluster,dominant_trajectory_domain,structural_cluster,shock_sensitivity,driver_1,driver_2,driver_3,driver_1_direction,driver_2_direction,driver_3_direction,data_reliability,country_archetype
0,Algeria,T0_reference,Education-Led Social Improvers,Labor,North African Reformers,-0.087545,NaN,NaN,NaN,NaN,NaN,NaN,Medium,Transitional or Mixed Trajectory System
1,Bahrain,T0_reference,Steady Reformers,Economy,High-Income Empowerers,-0.134814,NaN,NaN,NaN,NaN,NaN,NaN,High,Stable Reformers with Institutional Momentum
2,"Egypt, Arab Rep.",T0_reference,Education-Led Social Improvers,WBL,Fragile States,-0.149559,NaN,NaN,NaN,NaN,NaN,NaN,Medium-Low,Structurally Fragile but Statistically Buffered
3,"Iran, Islamic Rep.",T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.120607,NaN,NaN,NaN,NaN,NaN,NaN,Medium-Low,Structurally Fragile but Statistically Buffered
4,Iraq,T0_reference,Education-Led Social Improvers,Education,Fragile States,-0.125821,NaN,NaN,NaN,NaN,NaN,NaN,Low,Structurally Fragile but Statistically Buffered
5,Jordan,T0_reference,Steady Moderate Improvers,Governance,Fragile States,-0.153246,NaN,NaN,NaN,NaN,NaN,NaN,High,Stable Reformers with Institutional Momentum
6,Kuwait,T0_reference,Steady Moderate Improvers,Health,High-Income Empowerers,-0.112110,NaN,NaN,NaN,NaN,NaN,NaN,Medium,Transitional or Mixed Trajectory System
7,Lebanon,T3_strong,Education-Led Social Improvers,Economy,Lebanon Outlier,0.089973,cat_Demographics_cagr,cat_LegalAutonomy_coef_variation,regional_deviation_std_mean,protective,risk_increasing,risk_increasing,Medium-Low,Shock-Exposed Fragile System
8,Libya,T0_reference,Education-Led Social Improvers,Education,North African Reformers,-0.068374,NaN,NaN,NaN,NaN,NaN,NaN,Collapsed,Collapsed Statistical System (False Stability)
9,Morocco,T3_strong,Morocco,Health,North African Reformers,0.015850,cat_LegalAutonomy_coef_variation,cat_Economy_trend_slope,coef_variation_mean,risk_increasing,risk_increasing,risk_increasing,High,Reform–Shock Collision State


## Create Narratives

In [115]:
def generate_country_narrative(row):
    country = row["country"]
    archetype = row["country_archetype"]
    risk = row["risk_tier"]
    structure = row["structural_cluster"]
    trajectory = row["dominant_trajectory_domain"]
    shock = row["shock_sensitivity"]
    reliability = row["data_reliability"]

    # ---- Archetype-specific narratives ----
    
    if archetype == "Collapsed Statistical System (False Stability)":
        return (
            f"{country} is classified as a **Collapsed Statistical System**, meaning that "
            f"apparent stability in model outputs reflects **data-system failure rather than real-world resilience**. "
            f"Despite severe structural stress and historical instability, the dataset shows muted trajectories and "
            f"low shock sensitivity, consistent with **collapsed reporting capacity** ({reliability} reliability). "
            f"As a result, classifier-based risk estimates for {country} are **non-interpretable**, and conclusions "
            f"should rely on qualitative or external evidence rather than quantitative indicators."
        )

    if archetype == "Shock-Exposed Fragile System":
        return (
            f"{country} is classified as a **Shock-Exposed Fragile System**, characterized by "
            f"high model-estimated risk ({risk}), structural fragility ({structure}), and "
            f"elevated shock sensitivity. The country’s dominant trajectory in the **{trajectory}** domain "
            f"shows strong volatility, indicating that external and internal shocks translate directly into "
            f"instability rather than being absorbed institutionally. Given {reliability} data reliability, "
            f"these signals likely reflect **real compounding vulnerability**, making this a priority case for "
            f"policy intervention and crisis mitigation."
        )

    if archetype == "Reform–Shock Collision State":
        return (
            f"{country} is classified as a **Reform–Shock Collision State**, where elevated risk ({risk}) "
            f"emerges despite strong reform-oriented dynamics. Although structurally positioned among "
            f"{structure}, the country exhibits positive but unstable trajectories in the **{trajectory}** domain, "
            f"combined with measurable shock sensitivity. With {reliability} data reliability, the model captures "
            f"a pattern where **reform momentum collides with economic or social stress**, generating risk not from "
            f"policy absence but from reform–capacity misalignment."
        )

    if archetype == "Structurally Fragile but Statistically Buffered":
        return (
            f"{country} is classified as **Structurally Fragile but Statistically Buffered**. While the classifier "
            f"assigns a low-risk label ({risk}), the country remains embedded in a fragile structural cluster "
            f"({structure}). Shock sensitivity is muted, and dominant trajectories in the **{trajectory}** domain "
            f"appear smooth, a pattern consistent with **data smoothing or delayed reporting** ({reliability} reliability). "
            f"This suggests that low estimated risk should be interpreted cautiously, as it may mask latent instability."
        )

    if archetype == "Education-Led Improvers with Latent Risk":
        return (
            f"{country} is classified as an **Education-Led Improver with Latent Risk**. The country shows steady "
            f"progress driven primarily by improvements in the **{trajectory}** domain, contributing to its low "
            f"classified risk ({risk}). However, reliance on human-capital gains without parallel institutional "
            f"strengthening may leave the system vulnerable to future shocks. Given {reliability} data reliability, "
            f"this trajectory reflects genuine progress but also highlights the importance of complementary reforms."
        )

    if archetype == "Stable Reformers with Institutional Momentum":
        return (
            f"{country} is classified as a **Stable Reformer with Institutional Momentum**, reflecting "
            f"low estimated risk ({risk}), strong structural positioning ({structure}), and negative or neutral "
            f"shock sensitivity. Dominant trajectories in the **{trajectory}** domain remain stable over time, "
            f"indicating effective shock absorption and policy continuity. With {reliability} data reliability, "
            f"this profile represents one of the most resilient gender-policy environments in the region."
        )

    # ---- Fallback ----
    return (
        f"{country} is classified as a **Transitional or Mixed Trajectory System**, exhibiting a combination of "
        f"structural, dynamic, and shock-response characteristics that do not align cleanly with a single risk pattern. "
        f"While classified as {risk}, its dominant **{trajectory}** trajectory and {reliability} data reliability "
        f"suggest that further qualitative investigation is required for precise interpretation."
    )


In [116]:
final_cross_model_matrix["country_narrative"] = final_cross_model_matrix.apply(
    generate_country_narrative,
    axis=1
)


In [117]:
final_cross_model_matrix[
    final_cross_model_matrix["country"].isin(["Lebanon", "Libya", "Morocco", "Qatar"])
][["country", "country_archetype", "country_narrative"]]


,country,country_archetype,country_narrative
7,Lebanon,Shock-Exposed Fragile System,Lebanon is classified as a **Shock-Exposed Fra...
8,Libya,Collapsed Statistical System (False Stability),Libya is classified as a **Collapsed Statistic...
9,Morocco,Reform–Shock Collision State,Morocco is classified as a **Reform–Shock Coll...
11,Qatar,Stable Reformers with Institutional Momentum,Qatar is classified as a **Stable Reformer wit...


In [119]:
final_cross_model_matrix.to_csv(
    "/Users/daniehbenotman/Desktop/worldbank_tableau_project/analysis/synthesis/cross_model_with_narratives.csv",
    index=False
)


## Policy summary generator

In [120]:
def generate_policy_summary(row):
    country = row["country"]
    archetype = row["country_archetype"]
    trajectory = row["dominant_trajectory_domain"]
    shock = row["shock_sensitivity"]
    reliability = row["data_reliability"]
    risk = row["risk_tier"]

    if archetype == "Collapsed Statistical System (False Stability)":
        return (
            "Quantitative indicators are unreliable due to data-system collapse; "
            "policy assessment should prioritize qualitative evidence and humanitarian monitoring."
        )

    if archetype == "Shock-Exposed Fragile System":
        return (
            "High-risk environment with direct exposure to shocks; "
            f"interventions should focus on stabilizing the {trajectory.lower()} domain and strengthening crisis-response capacity."
        )

    if archetype == "Reform–Shock Collision State":
        return (
            "Reform momentum is present but vulnerable to disruption; "
            f"policy should prioritize institutional absorption capacity in the {trajectory.lower()} domain."
        )

    if archetype == "Structurally Fragile but Statistically Buffered":
        return (
            "Low measured risk may mask latent instability; "
            f"preventive reforms in the {trajectory.lower()} domain are recommended despite muted indicators."
        )

    if archetype == "Education-Led Improvers with Latent Risk":
        return (
            "Progress driven by human-capital gains; "
            "complementary institutional and governance reforms are needed to sustain long-term resilience."
        )

    if archetype == "Stable Reformers with Institutional Momentum":
        return (
            "Low-risk and resilient system; "
            f"policy focus should shift from crisis mitigation to deepening reforms in the {trajectory.lower()} domain."
        )

    return (
        "Mixed signals across risk, structure, and dynamics; "
        "targeted diagnostic assessment is recommended before policy prioritization."
    )


In [121]:
policy_brief = final_cross_model_matrix[[
    "country",
    "country_archetype",
    "risk_tier",
    "dominant_trajectory_domain",
    "shock_sensitivity",
    "data_reliability"
]].copy()

policy_brief["policy_summary"] = policy_brief.apply(
    generate_policy_summary,
    axis=1
)


In [122]:
policy_brief

,country,country_archetype,risk_tier,dominant_trajectory_domain,shock_sensitivity,data_reliability,policy_summary
0,Algeria,Transitional or Mixed Trajectory System,T0_reference,Labor,-0.087545,Medium,"Mixed signals across risk, structure, and dyna..."
1,Bahrain,Stable Reformers with Institutional Momentum,T0_reference,Economy,-0.134814,High,Low-risk and resilient system; policy focus sh...
2,"Egypt, Arab Rep.",Structurally Fragile but Statistically Buffered,T0_reference,WBL,-0.149559,Medium-Low,Low measured risk may mask latent instability;...
3,"Iran, Islamic Rep.",Structurally Fragile but Statistically Buffered,T0_reference,Education,-0.120607,Medium-Low,Low measured risk may mask latent instability;...
4,Iraq,Structurally Fragile but Statistically Buffered,T0_reference,Education,-0.125821,Low,Low measured risk may mask latent instability;...
5,Jordan,Stable Reformers with Institutional Momentum,T0_reference,Governance,-0.153246,High,Low-risk and resilient system; policy focus sh...
6,Kuwait,Transitional or Mixed Trajectory System,T0_reference,Health,-0.112110,Medium,"Mixed signals across risk, structure, and dyna..."
7,Lebanon,Shock-Exposed Fragile System,T3_strong,Economy,0.089973,Medium-Low,High-risk environment with direct exposure to ...
8,Libya,Collapsed Statistical System (False Stability),T0_reference,Education,-0.068374,Collapsed,Quantitative indicators are unreliable due to ...
9,Morocco,Reform–Shock Collision State,T3_strong,Health,0.015850,High,Reform momentum is present but vulnerable to d...


In [ ]:
policy_brief.to_csv(
    "/Users/daniehbenotman/Desktop/worldbank_tableau_project/analysis/analysis_outputs/country_policy_brief_summary.csv",
    index=False
)
